# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We demonstrate step-by-step how to access Croissant metadata, overview record sets, extract records, and process them for exploratory analysis and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

# Display dataset name and description
print(f"Dataset Name: {metadata_obj.name}\n")
print(f"Description: {metadata_obj.description}\n")

## 2. Data Overview

Let's look at the available record sets and fields, referencing them by their `@id`s as per the Croissant schema. We'll enumerate all record set `@id`s, their fields, and show a sample record from each.

*Note: If you don't see record sets or fields, check the Croissant schema for details or access the full metadata through `dataset.metadata`.*

In [ ]:
# List all record sets by their @id
record_sets = []
fields_per_record_set = {}

# dataset.metadata.recordSet may be empty or contain record sets; let's try to enumerate them
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    for rs in dataset.metadata.recordSet:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs)
        record_sets.append(rs_id)
        # List fields for this record set
        fields = rs.get('field', []) if isinstance(rs, dict) else []
        fields_ids = [f['@id'] if isinstance(f, dict) else str(f) for f in fields]
        fields_per_record_set[rs_id] = fields_ids
else:
    # Fallback: Try to enumerate available record sets via `dataset.record_sets`
    try:
        record_sets = [r['@id'] for r in dataset.record_sets]
        for rs in dataset.record_sets:
            rs_id = rs['@id']
            fields_ids = [f['@id'] for f in rs.get('field', [])]
            fields_per_record_set[rs_id] = fields_ids
    except Exception:
        # Try to access internal dataset._record_sets if public accessor is unavailable
        record_sets = [r['@id'] for r in getattr(dataset, '_record_sets', [])]
        for rs in getattr(dataset, '_record_sets', []):
            rs_id = rs['@id']
            fields_ids = [f['@id'] for f in rs.get('field', [])]
            fields_per_record_set[rs_id] = fields_ids

print("Record Sets Detected:")
for rs_id in record_sets:
    print(f"- {rs_id} (fields: {fields_per_record_set.get(rs_id, [])})")

# Preview sample records from each record set by @id
for rs_id in record_sets:
    print(f"\nSample records from record set @id: {rs_id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

## 3. Data Extraction

We load records from each record set into DataFrames for further analysis. Record sets and fields are referenced by their `@id`. Choose a record set to analyze—modify as needed if you wish to explore another set.

In [ ]:
# Extract data from each record set into a DataFrame
dataframes = {}

for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"DataFrame created for record set {rs_id}, columns: {dataframes[rs_id].columns.tolist()}")
            print(dataframes[rs_id].head())
        else:
            print(f"No records found for record set {rs_id}.")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Select default record set to work with (first detected, or specify manually)
chosen_record_set_id = record_sets[0] if record_sets else None
df = dataframes.get(chosen_record_set_id, pd.DataFrame())

## 4. Exploratory Data Analysis (EDA)

We'll process numeric and categorical fields using their `@id`. Common operations include filtering, normalization, and grouping. Update field `@id`s as appropriate—refer to the columns printed above.

In [ ]:
# Choose a numeric field and a group field by their @id
if not df.empty:
    # Show all column names for selection
    print("Detected columns:", df.columns.tolist())
    numeric_field_id = None
    group_field_id = None

    # Try to auto-select a numeric field and a group field
    for col in df.columns:
        # Guess numeric fields
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Guess a group field (categorical)
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break

    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")

    # Set threshold for filtering
    threshold = df[numeric_field_id].median() if numeric_field_id else None

    # Filter records
    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df.copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    if numeric_field_id:
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group data by group field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No data available for EDA. Please check the record sets and records extraction above.")

## 5. Visualization

Visualize numeric distributions or relationships between selected fields. Here, we plot the normalized numeric field distribution and compare across groups, using field `@id`s detected above.

In [ ]:
# Plotting numeric field distribution and by group
if not df.empty and numeric_field_id:
    plt.figure(figsize=(8, 4))
    filtered_df[f"{numeric_field_id}_normalized"].hist(bins=30, alpha=0.7, label=f"{numeric_field_id}_normalized")
    plt.title(f"Normalized Distribution of {numeric_field_id} (@id)")
    plt.xlabel(f"{numeric_field_id} normalized")
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

    if group_field_id:
        # Plot group means for numeric field
        plt.figure(figsize=(8, 4))
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No data available for visualization. Please check previous steps.")

## 6. Conclusion

This notebook demonstrated loading, processing, and visualizing the FAIR^2 dataset using `mlcroissant`. Key steps were:
- Accessing metadata and records by their `@id`.
- Extracting DataFrames from Croissant-structured records.
- Filtering and normalizing numeric fields for EDA.
- Grouping and visualizing relationships between fields.

Further exploration can be tailored by selecting specific record sets and fields based on their `@id` from the Croissant schema. Review the dataset source and documentation for full contextual understanding.

<!-- End of notebook -->